# Section-level article-bootstrap confidence intervals

This notebook calculates section-level point estimates and 95% confidence intervals while resampling whole articles.

- Use `PRED_COL = "y_pred"` for local predictions.
- Use `PRED_COL = "global_y_pred"` for post-decoder predictions.
- Confidence intervals are calculated for accuracy and macro-F1.
- Macro-precision and macro-recall are also reported as point estimates.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# # For text-based component model, use:
# MODEL_RUN_NAME = (
#     "text_pubmedbert_k5_late_prob_timeline_logreg"
# )

# For Text + Progression, use:
MODEL_RUN_NAME = (
    "text_plus_progression_pubmedbert_k5_late_prob_timeline_logreg"
)

PREDICTION_FILE = (
    Path("results")
    / "test"
    / "global_layout_prior"
    / MODEL_RUN_NAME
    / "predictions.csv"
)

OUTPUT_FILE = (
    PREDICTION_FILE.parent
    / "section_level_bootstrap_ci.csv"
)

# # For text-based component model, use:
# MODEL_NAME = "Text + Global Prior"

# For Text + Progression, use:
MODEL_NAME = "Text + Progression + Global Prior"

PAPER_COL = "paper_id"
TRUE_COL = "y_true"

# Use "global_y_pred" for post-decoder predictions.
# Use "y_pred" only for local predictions.
PRED_COL = "global_y_pred"

N_BOOTSTRAP = 5000
RANDOM_SEED = 42
CI_LEVEL = 0.95

LABELS = ["I", "M", "R", "D", "NoIMRaD"]

print("Prediction file:", PREDICTION_FILE)
print("Output file:", OUTPUT_FILE)

In [ ]:
# --------------------------------------------------
# Load and validate predictions
# --------------------------------------------------
df = pd.read_csv(PREDICTION_FILE)

required_cols = {PAPER_COL, TRUE_COL, PRED_COL}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

if df[[PAPER_COL, TRUE_COL, PRED_COL]].isna().any().any():
    raise ValueError(
        f"The prediction file contains missing values in "
        f"{[PAPER_COL, TRUE_COL, PRED_COL]}."
    )

unknown_true = sorted(set(df[TRUE_COL]) - set(LABELS))
unknown_pred = sorted(set(df[PRED_COL]) - set(LABELS))

if unknown_true:
    raise ValueError(f"Unexpected true labels: {unknown_true}")
if unknown_pred:
    raise ValueError(f"Unexpected predicted labels: {unknown_pred}")

print(f"Model: {MODEL_NAME}")
print(f"Prediction column: {PRED_COL}")
print(f"Section blocks: {len(df):,}")
print(f"Articles: {df[PAPER_COL].nunique():,}")


In [ ]:
# --------------------------------------------------
# Point estimates
# --------------------------------------------------
y_true = df[TRUE_COL].to_numpy()
y_pred = df[PRED_COL].to_numpy()

accuracy = accuracy_score(y_true, y_pred)
macro_precision = precision_score(
    y_true,
    y_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
macro_recall = recall_score(
    y_true,
    y_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)

print("Point estimates")
print(f"Accuracy:        {accuracy:.6f}")
print(f"Macro-Precision: {macro_precision:.6f}")
print(f"Macro-Recall:    {macro_recall:.6f}")
print(f"Macro-F1:        {macro_f1:.6f}")


In [ ]:
# --------------------------------------------------
# Article-level bootstrap
# --------------------------------------------------
rng = np.random.default_rng(RANDOM_SEED)

paper_ids = df[PAPER_COL].drop_duplicates().to_numpy()
n_papers = len(paper_ids)

# Store row indices for each article. Sampling an article more than once
# repeats all of its section rows, preserving bootstrap multiplicity.
paper_row_indices = {
    paper_id: group.index.to_numpy()
    for paper_id, group in df.groupby(PAPER_COL, sort=False)
}

bootstrap_accuracy = np.empty(N_BOOTSTRAP, dtype=float)
bootstrap_macro_f1 = np.empty(N_BOOTSTRAP, dtype=float)

for b in range(N_BOOTSTRAP):
    sampled_papers = rng.choice(
        paper_ids,
        size=n_papers,
        replace=True,
    )

    sampled_indices = np.concatenate(
        [paper_row_indices[paper_id] for paper_id in sampled_papers]
    )

    sampled_true = df.loc[sampled_indices, TRUE_COL].to_numpy()
    sampled_pred = df.loc[sampled_indices, PRED_COL].to_numpy()

    bootstrap_accuracy[b] = accuracy_score(
        sampled_true,
        sampled_pred,
    )

    bootstrap_macro_f1[b] = f1_score(
        sampled_true,
        sampled_pred,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )

print(f"Completed {N_BOOTSTRAP:,} article-level bootstrap samples.")


In [ ]:
# --------------------------------------------------
# Percentile confidence intervals
# --------------------------------------------------
alpha = 1.0 - CI_LEVEL
percentiles = [
    100.0 * alpha / 2.0,
    100.0 * (1.0 - alpha / 2.0),
]

accuracy_ci = np.percentile(
    bootstrap_accuracy,
    percentiles,
)
macro_f1_ci = np.percentile(
    bootstrap_macro_f1,
    percentiles,
)

print(f"{int(CI_LEVEL * 100)}% article-level bootstrap confidence intervals")
print(
    f"Accuracy: {accuracy:.6f} "
    f"[{accuracy_ci[0]:.6f}, {accuracy_ci[1]:.6f}]"
)
print(
    f"Macro-F1: {macro_f1:.6f} "
    f"[{macro_f1_ci[0]:.6f}, {macro_f1_ci[1]:.6f}]"
)


In [ ]:
# --------------------------------------------------
# Save summary
# --------------------------------------------------
summary = pd.DataFrame(
    {
        "model": [MODEL_NAME, MODEL_NAME],
        "prediction_column": [PRED_COL, PRED_COL],
        "metric": ["accuracy", "macro_f1"],
        "estimate": [accuracy, macro_f1],
        "ci_lower": [accuracy_ci[0], macro_f1_ci[0]],
        "ci_upper": [accuracy_ci[1], macro_f1_ci[1]],
        "n_bootstrap": [N_BOOTSTRAP, N_BOOTSTRAP],
        "bootstrap_unit": ["article", "article"],
        "n_articles": [n_papers, n_papers],
        "n_section_blocks": [len(df), len(df)],
        "random_seed": [RANDOM_SEED, RANDOM_SEED],
    }
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_FILE, index=False)

print(f"Saved: {OUTPUT_FILE}")
display(summary)
